# RAS 2026 PSC — RL-guided Large Neighborhood Search

**Pipeline:** Greedy → BC warm-start → LNS-RL (RL-guided perturbation)

**Paper angle:** *RL-guided Large Neighborhood Search for Railroad Blocking Plan Optimization*

**MDP formulation:**
- State: current blocking plan + per-demand features
- Action (step 1): which demand to perturb (selection policy)
- Action (step 2): which route to assign (routing policy)
- Reward: immediate stress score improvement (dense signal)
- Episode: T perturbation steps starting from greedy solution

## 0. Setup

In [ ]:
import os
if not os.path.exists('ras2026'):
    !git clone https://github.com/nepersoned/ras2026.git
os.chdir('ras2026')
print('cwd:', os.getcwd())

In [ ]:
!pip install -q scipy numpy pandas torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Google Drive 폴더 ID로 직접 접근
DRIVE_DATA = '/content/drive/MyDrive/ras_release_v1.1'
print('Data path:', DRIVE_DATA)
print('Exists:', os.path.exists(DRIVE_DATA))
print(os.listdir(DRIVE_DATA) if os.path.exists(DRIVE_DATA) else 'NOT FOUND')

In [ ]:
# Drive 데이터를 로컬로 심볼릭 링크 (solver.py가 기대하는 경로)
import os
if not os.path.exists('ras_release_v1.1'):
    os.symlink(DRIVE_DATA, 'ras_release_v1.1')
print('Ready:', os.path.exists('ras_release_v1.1/ras_release_v1.1/datasets/l1/node.csv'))

## 1. Environment

In [ ]:
import sys
sys.path.insert(0, '.')

import math, random, time, json, csv
from pathlib import Path
from collections import defaultdict
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

from solver import (
    load_layer, load_od_matrix, Network, Demand, Solution,
    greedy_construct, evaluate, build_json,
    COMMODITY_TO_BLOCK_TYPE, DIRECT_ONLY,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
BASE = Path('ras_release_v1.1/ras_release_v1.1')

In [ ]:
# ── RAS MDP Environment ────────────────────────────────────────────────────────

BLOCK_TYPES = ['Merchandise', 'Coal', 'Grain', 'Intermodal', 'Automobile']
MAX_HUBS    = 50
FEAT_DIM    = 7 + MAX_HUBS * 4   # 207
N_ACTIONS   = MAX_HUBS + 2       # 0=direct, 1..MAX_HUBS=hub, MAX_HUBS+1=unserved


def load_env(layer: str, multiplier: float):
    """Load all data and precompute network for one scenario."""
    nodes_df, links_df, demands_scaled, demands_raw, settings = \
        load_layer(layer, multiplier)

    yard_rows = nodes_df[nodes_df['node_type'] == 'yard']
    yard_info = {
        int(r['node_id']): {
            'num_tracks':        float(r.get('num_tracks',        9999) or 9999),
            'handling_capacity': float(r.get('handling_capacity', 1e9)  or 1e9),
            'handling_cost':     float(r.get('handling_cost',     0)    or 0),
        }
        for _, r in yard_rows.iterrows()
    }

    origin_ids   = set(demands_scaled['origin_yard_id'].astype(int))
    dest_ids     = set(demands_scaled['dest_yard_id'].astype(int))
    all_yard_ids = sorted(origin_ids | dest_ids)

    all_od_pairs = {(o, d) for o in all_yard_ids for d in all_yard_ids if o != d}
    od_matrix    = load_od_matrix(all_od_pairs)

    net = Network(nodes_df, links_df, origin_ids, dest_ids, settings, verbose=True)

    demands = [
        Demand(
            idx            = idx,
            demand_id      = int(row['demand_id']),
            commodity_type = str(row['block_type']),
            block_type     = COMMODITY_TO_BLOCK_TYPE.get(str(row['block_type']), 'Manifest'),
            origin         = int(row['origin_yard_id']),
            dest           = int(row['dest_yard_id']),
            volume         = int(row['volume']),
            sp_dist        = od_matrix.get(
                (int(row['origin_yard_id']), int(row['dest_yard_id'])),
                net.dist(int(row['origin_yard_id']), int(row['dest_yard_id']))
            ),
        )
        for idx, (_, row) in enumerate(demands_scaled.iterrows())
    ]

    return dict(
        net=net, od_matrix=od_matrix, settings=settings,
        yard_info=yard_info, demands=demands,
        all_yards=all_yard_ids,
        nodes_df=nodes_df, links_df=links_df, demands_raw=demands_raw,
    )

## 2. Feature Engineering

In [ ]:
def ranked_hubs(dem, env):
    if dem.commodity_type in DIRECT_ONLY:
        return []
    scored = []
    for hub in env['all_yards']:
        if hub == dem.origin or hub == dem.dest:
            continue
        d1 = env['net'].dist(dem.origin, hub)
        d2 = env['net'].dist(hub, dem.dest)
        if math.isinf(d1) or math.isinf(d2):
            continue
        scored.append((d1 + d2, hub))
    scored.sort()
    return [h for _, h in scored]


def demand_feature(dem, hubs, env):
    """207-dim feature vector for one demand."""
    od_d  = env['net'].dist(dem.origin, dem.dest)
    od_ic = env['net'].interchanges(dem.origin, dem.dest)
    ct_oh = [1.0 if dem.commodity_type == t else 0.0 for t in BLOCK_TYPES]
    base  = [math.log1p(dem.volume), math.log1p(od_d), od_ic / 5.0] + ct_oh  # 8 dims... wait
    # actually: log_vol, log_od_dist, od_ic_norm, 4×block_type_onehot → 7 total
    bt_oh = [1.0 if COMMODITY_TO_BLOCK_TYPE.get(dem.commodity_type,'Manifest') == t
             else 0.0
             for t in ['Manifest','Bulk','Intermodal','Multilevel']]
    base  = [math.log1p(dem.volume), math.log1p(od_d if math.isfinite(od_d) else 1e6),
             od_ic / 5.0] + bt_oh   # 7 dims

    hub_feats = [0.0] * (MAX_HUBS * 4)
    for j, hub in enumerate(hubs[:MAX_HUBS]):
        d1  = env['net'].dist(dem.origin, hub)
        d2  = env['net'].dist(hub, dem.dest)
        if math.isinf(d1) or math.isinf(d2):
            continue
        ic1 = env['net'].interchanges(dem.origin, hub)
        ic2 = env['net'].interchanges(hub, dem.dest)
        hc  = env['yard_info'].get(hub, {}).get('handling_cost', 0.0)
        hub_feats[j*4]   = math.log1p(d1)
        hub_feats[j*4+1] = math.log1p(d2)
        hub_feats[j*4+2] = (ic1 + ic2) / 5.0
        hub_feats[j*4+3] = hc / 500.0

    return base + hub_feats   # 207 dims


def action_mask(dem, hubs, env):
    """Boolean mask of valid actions."""
    mask = [False] * N_ACTIONS
    if not math.isinf(env['net'].dist(dem.origin, dem.dest)):
        mask[0] = True
    if dem.commodity_type not in DIRECT_ONLY:
        for j, hub in enumerate(hubs[:MAX_HUBS]):
            d1 = env['net'].dist(dem.origin, hub)
            d2 = env['net'].dist(hub, dem.dest)
            if not math.isinf(d1) and not math.isinf(d2):
                mask[j + 1] = True
    mask[N_ACTIONS - 1] = True  # always can be unserved
    return mask


def action_to_route(action, dem, hubs):
    if action == 0:
        return [(dem.origin, dem.dest)]
    if action == N_ACTIONS - 1:
        return None
    hi = action - 1
    if hi >= len(hubs):
        return None
    return [(dem.origin, hubs[hi]), (hubs[hi], dem.dest)]


def precompute(env):
    """Precompute per-demand features, masks, and hub lists."""
    data = []
    for dem in env['demands']:
        if dem.volume <= 0:
            data.append(None)
            continue
        hubs = ranked_hubs(dem, env)
        feat = demand_feature(dem, hubs, env)
        mask = action_mask(dem, hubs, env)
        data.append({'feat': feat, 'mask': mask, 'hubs': hubs})
    return data


print('Feature engineering ready.')

## 3. Policy Network

In [ ]:
class BlockingPolicy(nn.Module):
    """MLP policy: demand features → action logits.

    For paper: this is a shared-parameter policy over all demands.
    Input: 207-dim demand feature vector
    Output: N_ACTIONS=52 logits (direct / hub_1..hub_50 / unserved)
    """
    def __init__(self, feat_dim=FEAT_DIM, n_actions=N_ACTIONS, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        # x: (B, FEAT_DIM)  →  (B, N_ACTIONS)
        return self.net(x)


def make_policy():
    return BlockingPolicy().to(DEVICE)


print(f'Policy: {FEAT_DIM} → 256 → 256 → {N_ACTIONS}  ({sum(p.numel() for p in make_policy().parameters()):,} params)')

## 4. Behavioral Cloning (warm-start)

In [ ]:
def greedy_action_label(route, hubs):
    """Convert greedy routing to action index for BC."""
    if route is None:
        return N_ACTIONS - 1
    if len(route) == 1:
        return 0
    hub = route[0][1]
    top = hubs[:MAX_HUBS]
    if hub in top:
        return top.index(hub) + 1
    return N_ACTIONS - 1


def pretrain_bc(rout_policy, env, dd, n_epochs=300, lr=1e-3, print_every=50):
    """Behavioral Cloning: warm-start routing policy from greedy solution."""
    init_sol = greedy_construct(env['demands'], env['net'], env['od_matrix'],
                                env['settings'], env['yard_info'])
    score0, _ = evaluate(init_sol, env['net'], env['od_matrix'],
                         env['settings'], env['yard_info'])

    feats, masks, labels = [], [], []
    for i, dem in enumerate(env['demands']):
        if dd[i] is None:
            continue
        route  = init_sol.routings[i]
        label  = greedy_action_label(route, dd[i]['hubs'])
        feats.append(dd[i]['feat'])
        masks.append(dd[i]['mask'])
        labels.append(label)

    feat_t  = torch.tensor(feats,  dtype=torch.float32, device=DEVICE)
    label_t = torch.tensor(labels, dtype=torch.long,    device=DEVICE)
    mask_t  = torch.tensor(masks,  dtype=torch.bool,    device=DEVICE)

    opt = torch.optim.Adam(rout_policy.parameters(), lr=lr)
    print(f'\n── BC warm-start ──  N={len(labels)}  greedy_score={score0:,.0f}')

    for ep in range(1, n_epochs + 1):
        rout_policy.train()
        logits = rout_policy(feat_t).masked_fill(~mask_t, -1e9)
        loss   = F.cross_entropy(logits, label_t)
        opt.zero_grad(); loss.backward(); opt.step()
        if ep % print_every == 0:
            with torch.no_grad():
                acc = (logits.argmax(1) == label_t).float().mean().item() * 100
            print(f'  ep {ep:4d} | loss={loss.item():.4f} | acc={acc:.1f}%')

    print('BC done.')
    return init_sol, score0

## 5. LNS-RL: Selection Policy + Routing Policy

In [ ]:
# ── Selection Policy ──────────────────────────────────────────────────────────

SEL_FEAT_DIM = FEAT_DIM + 3   # 207 static + [is_direct, is_hub, is_unserved]

class SelectionPolicy(nn.Module):
    """Given current solution state, pick which demand to perturb."""
    def __init__(self, feat_dim=SEL_FEAT_DIM, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)   # (N_demands,)


class RoutingPolicy(nn.Module):
    """demand features → which route to assign."""
    def __init__(self, feat_dim=FEAT_DIM, n_actions=N_ACTIONS, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )
    def forward(self, x):
        return self.net(x)


def make_lns_policies():
    return SelectionPolicy().to(DEVICE), RoutingPolicy().to(DEVICE)


# ── State features ────────────────────────────────────────────────────────────

def selection_features(dd, routings):
    feats, valid_idxs = [], []
    for i, d in enumerate(dd):
        if d is None:
            continue
        route = routings[i]
        is_direct   = 1.0 if (route is not None and len(route) == 1) else 0.0
        is_hub      = 1.0 if (route is not None and len(route) == 2) else 0.0
        is_unserved = 1.0 if route is None else 0.0
        feats.append(d['feat'] + [is_direct, is_hub, is_unserved])
        valid_idxs.append(i)
    return feats, valid_idxs


def evaluate_routing_change(env, old_sol, demand_idx, new_route):
    new_routings = list(old_sol.routings)
    new_routings[demand_idx] = new_route
    new_sol = Solution(env['demands'], new_routings)
    new_score, _ = evaluate(new_sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])
    return new_score, new_sol


# ── LNS-RL Episode with SA acceptance ────────────────────────────────────────

def lns_episode(sel_policy, rout_policy, env, dd, init_sol, init_score,
                T_steps=50, temperature=1.0, sa_temp=None, greedy=False):
    """
    LNS-RL episode.
    - RL picks which demand to perturb (selection) and which route (routing).
    - SA acceptance criterion: always accept improvements;
      accept worsening moves with prob exp(-delta / sa_temp).
    sa_temp=None → always accept (pure RL exploration).
    """
    sel_policy.eval(); rout_policy.eval()

    sol        = init_sol
    score      = init_score
    best_score = score
    best_sol   = sol
    transitions = []

    for _ in range(T_steps):
        # Step 1: RL selection
        sel_feats, valid_idxs = selection_features(dd, sol.routings)
        if not valid_idxs:
            break
        sel_feat_t = torch.tensor(sel_feats, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            sel_scores = sel_policy(sel_feat_t)
            j = sel_scores.argmax().item() if greedy \
                else Categorical(logits=sel_scores / temperature).sample().item()
        i = valid_idxs[j]

        # Step 2: RL routing
        rout_feat = dd[i]['feat']
        rout_mask = dd[i]['mask']
        rout_feat_t = torch.tensor([rout_feat], dtype=torch.float32, device=DEVICE)
        rout_mask_t = torch.tensor([rout_mask], dtype=torch.bool,    device=DEVICE)
        with torch.no_grad():
            rout_logits = rout_policy(rout_feat_t).masked_fill(~rout_mask_t, -1e9)
            a = rout_logits.argmax(dim=1).item() if greedy \
                else Categorical(logits=rout_logits[0] / temperature).sample().item()
        new_route = action_to_route(a, env['demands'][i], dd[i]['hubs'])

        # Evaluate
        new_score, new_sol = evaluate_routing_change(env, sol, i, new_route)
        delta  = new_score - score
        reward = -delta / max(abs(score), 1.0)   # positive = improvement

        # SA acceptance
        if delta < 0:
            accept = True   # always accept improvements
        elif sa_temp is not None and sa_temp > 0:
            accept = random.random() < math.exp(-delta / sa_temp)
        else:
            accept = True   # no SA → always accept

        transitions.append({
            'sel_feat':    sel_feats, 'valid_idxs': valid_idxs, 'sel_j': j,
            'rout_feat':   rout_feat, 'rout_mask':  rout_mask,  'rout_action': a,
            'reward':      reward,    'accepted':    accept,
        })

        if accept:
            sol   = new_sol
            score = new_score
        if score < best_score:
            best_score = score
            best_sol   = sol

    return transitions, best_sol, best_score


# ── REINFORCE update ──────────────────────────────────────────────────────────

def lns_update(sel_policy, rout_policy, opt_sel, opt_rout, transitions,
               gamma=0.99, entropy_coef=0.01):
    if not transitions:
        return 0.0, 0.0

    R = 0.0
    returns = []
    for t in reversed(transitions):
        R = t['reward'] + gamma * R
        returns.insert(0, R)
    returns_t = torch.tensor(returns, dtype=torch.float32, device=DEVICE)
    returns_t = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)

    sel_loss_total  = torch.tensor(0.0, device=DEVICE)
    rout_loss_total = torch.tensor(0.0, device=DEVICE)
    sel_policy.train(); rout_policy.train()

    for t, G in zip(transitions, returns_t):
        sf  = torch.tensor(t['sel_feat'],  dtype=torch.float32, device=DEVICE)
        sd  = Categorical(logits=sel_policy(sf))
        sl  = sd.log_prob(torch.tensor(t['sel_j'],      device=DEVICE))
        sel_loss_total = sel_loss_total + (-sl * G - entropy_coef * sd.entropy())

        rf  = torch.tensor([t['rout_feat']], dtype=torch.float32, device=DEVICE)
        rm  = torch.tensor([t['rout_mask']], dtype=torch.bool,    device=DEVICE)
        rl_ = rout_policy(rf).masked_fill(~rm, -1e9)
        rd  = Categorical(logits=rl_[0])
        rll = rd.log_prob(torch.tensor(t['rout_action'], device=DEVICE))
        rout_loss_total = rout_loss_total + (-rll * G - entropy_coef * rd.entropy())

    opt_sel.zero_grad();  sel_loss_total.backward();  torch.nn.utils.clip_grad_norm_(sel_policy.parameters(),  1.0); opt_sel.step()
    opt_rout.zero_grad(); rout_loss_total.backward(); torch.nn.utils.clip_grad_norm_(rout_policy.parameters(), 1.0); opt_rout.step()

    return sel_loss_total.item(), rout_loss_total.item()


print(f'LNS-RL (+ SA acceptance) ready.')
print(f'  Selection: {SEL_FEAT_DIM} → 128 → 128 → 1  ({sum(p.numel() for p in SelectionPolicy().parameters()):,} params)')
print(f'  Routing:   {FEAT_DIM} → 256 → 256 → {N_ACTIONS}  ({sum(p.numel() for p in RoutingPolicy().parameters()):,} params)')

## 6. LNS-RL Training: L1 → L2 → L3

In [ ]:
CASE_ORDER = [
    ('l1', 0.5), ('l1', 1.0), ('l1', 2.0),
    ('l2', 0.5), ('l2', 1.0), ('l2', 2.0),
    ('l3', 0.5), ('l3', 1.0), ('l3', 2.0),
]

# Hyperparams (tune for paper ablation)
BC_EPOCHS     = 300
LNS_EPISODES  = {'l1': 500, 'l2': 200, 'l3': 100}   # outer episodes
LNS_T_STEPS   = 50                                    # perturbation steps per episode
LR_SEL        = 3e-4
LR_ROUT       = 1e-4
TRANSFER_EPS  = {'l2': 200, 'l3': 100}

solutions  = {}   # tag → json dict
train_logs = {}   # tag → log list

import os
os.makedirs('saved_models', exist_ok=True)

In [ ]:
def train_lns_rl(sel_policy, rout_policy, env, dd, init_sol, init_score,
                 n_episodes=500, T_steps=50, print_every=50,
                 lr_sel=3e-4, lr_rout=1e-4,
                 sa_temp_init=None, sa_temp_final=None):
    """
    LNS-RL main loop.
    - RL policy learns selection + routing.
    - SA acceptance: accept worsening moves with exp(-Δ/T_sa), T_sa annealed.
    - sa_temp_init=None → always accept (no SA).
    """
    opt_sel  = torch.optim.Adam(sel_policy.parameters(),  lr=lr_sel)
    opt_rout = torch.optim.Adam(rout_policy.parameters(), lr=lr_rout)

    best_sol   = init_sol
    best_score = init_score
    log = []

    print(f'\n── LNS-RL ──  init={init_score:,.0f}  episodes={n_episodes}  T_steps={T_steps}')
    if sa_temp_init:
        print(f'  SA temp: {sa_temp_init} → {sa_temp_final}')

    for ep in range(1, n_episodes + 1):
        frac = ep / n_episodes
        # RL temperature: explore early, exploit late
        rl_temp = max(0.1, 1.0 - frac * 0.9)
        # SA temperature: high (accept bad moves) → low (only accept improvements)
        if sa_temp_init is not None:
            sa_temp = sa_temp_init * ((sa_temp_final / sa_temp_init) ** frac)
        else:
            sa_temp = None

        transitions, ep_best_sol, ep_best_score = lns_episode(
            sel_policy, rout_policy, env, dd,
            best_sol, best_score,
            T_steps=T_steps, temperature=rl_temp, sa_temp=sa_temp,
        )

        sel_loss, rout_loss = lns_update(
            sel_policy, rout_policy, opt_sel, opt_rout, transitions
        )

        if ep_best_score < best_score:
            best_score = ep_best_score
            best_sol   = ep_best_sol

        log.append({'ep': ep, 'ep_best': ep_best_score, 'best': best_score,
                    'sel_loss': sel_loss, 'rout_loss': rout_loss})

        if ep % print_every == 0:
            print(f'  ep {ep:4d} | ep_best={ep_best_score:>14,.0f} | '
                  f'best={best_score:>14,.0f} | rl_T={rl_temp:.2f}'
                  + (f' | sa_T={sa_temp:.1f}' if sa_temp else ''))

    print(f'  Final best: {best_score:,.0f}')
    return best_sol, best_score, log


# ── L1 Training ───────────────────────────────────────────────────────────────

import os
os.makedirs('saved_models', exist_ok=True)

for mult in [0.5, 1.0, 2.0]:
    tag = f'l1_{mult}'
    print(f'\n{"="*60}\n  L1 × {mult}\n{"="*60}')
    t0 = time.time()

    env = load_env('l1', mult)
    dd  = precompute(env)

    sel_policy, rout_policy = make_lns_policies()

    # Phase 1: BC warm-start
    init_sol, g0 = pretrain_bc(rout_policy, env, dd, n_epochs=BC_EPOCHS)
    init_score, _ = evaluate(init_sol, env['net'], env['od_matrix'],
                             env['settings'], env['yard_info'])

    # Phase 2: LNS-RL with SA acceptance
    best_sol, best_score, log = train_lns_rl(
        sel_policy, rout_policy, env, dd, init_sol, init_score,
        n_episodes=LNS_EPISODES['l1'], T_steps=LNS_T_STEPS,
        sa_temp_init=init_score * 0.05,   # start: accept ~5% worsening
        sa_temp_final=init_score * 0.001, # end: nearly greedy
    )
    train_logs[tag] = log

    score, stats = evaluate(best_sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])
    print(f'  stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')

    solutions[tag] = build_json(best_sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    torch.save({'sel': sel_policy.state_dict(), 'rout': rout_policy.state_dict()},
               f'saved_models/lns_l1_{mult}.pt')
    print(f'  Elapsed: {time.time()-t0:.1f}s')

In [ ]:
# ── L2 Training (transfer from L1 1.0×) ──────────────────────────────────────

for mult in [0.5, 1.0, 2.0]:
    tag = f'l2_{mult}'
    print(f'\n{"="*60}\n  L2 × {mult}\n{"="*60}')
    t0 = time.time()

    env = load_env('l2', mult)
    dd  = precompute(env)

    sel_policy, rout_policy = make_lns_policies()
    ckpt = torch.load('saved_models/lns_l1_1.0.pt', map_location=DEVICE)
    sel_policy.load_state_dict(ckpt['sel'])
    rout_policy.load_state_dict(ckpt['rout'])
    print('  Loaded L1 weights (transfer learning)')

    # Light BC fine-tune
    init_sol, g0 = pretrain_bc(rout_policy, env, dd, n_epochs=100, print_every=25)
    init_score, _ = evaluate(init_sol, env['net'], env['od_matrix'],
                             env['settings'], env['yard_info'])

    best_sol, best_score, log = train_lns_rl(
        sel_policy, rout_policy, env, dd, init_sol, init_score,
        n_episodes=TRANSFER_EPS['l2'], T_steps=LNS_T_STEPS,
        lr_sel=LR_SEL * 0.3, lr_rout=LR_ROUT * 0.3,
    )
    train_logs[tag] = log

    score, stats = evaluate(best_sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])
    print(f'  stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')

    solutions[tag] = build_json(best_sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    torch.save({'sel': sel_policy.state_dict(), 'rout': rout_policy.state_dict()},
               f'saved_models/lns_l2_{mult}.pt')
    print(f'  Elapsed: {time.time()-t0:.1f}s')

In [ ]:
# ── L3 Training (transfer from L2 1.0×) ──────────────────────────────────────

for mult in [0.5, 1.0, 2.0]:
    tag = f'l3_{mult}'
    print(f'\n{"="*60}\n  L3 × {mult}\n{"="*60}')
    t0 = time.time()

    env = load_env('l3', mult)
    dd  = precompute(env)

    sel_policy, rout_policy = make_lns_policies()
    ckpt = torch.load('saved_models/lns_l2_1.0.pt', map_location=DEVICE)
    sel_policy.load_state_dict(ckpt['sel'])
    rout_policy.load_state_dict(ckpt['rout'])
    print('  Loaded L2 weights (transfer learning)')

    init_sol, g0 = pretrain_bc(rout_policy, env, dd, n_epochs=50, print_every=10)
    init_score, _ = evaluate(init_sol, env['net'], env['od_matrix'],
                             env['settings'], env['yard_info'])

    best_sol, best_score, log = train_lns_rl(
        sel_policy, rout_policy, env, dd, init_sol, init_score,
        n_episodes=TRANSFER_EPS['l3'], T_steps=LNS_T_STEPS,
        lr_sel=LR_SEL * 0.1, lr_rout=LR_ROUT * 0.1,
    )
    train_logs[tag] = log

    score, stats = evaluate(best_sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])
    print(f'  stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')

    solutions[tag] = build_json(best_sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    torch.save({'sel': sel_policy.state_dict(), 'rout': rout_policy.state_dict()},
               f'saved_models/lns_l3_{mult}.pt')
    print(f'  Elapsed: {time.time()-t0:.1f}s')

## 7. Generate submission.csv

## 7. SA로 전체 9개 시나리오 풀고 제출

In [ ]:
rows = [['ID', 'data']]

for case_id, (layer, mult) in enumerate(CASE_ORDER):
    tag = f'{layer}_{mult}'
    sol = solutions.get(tag)
    if sol is None:
        print(f'[ID {case_id}] {tag} — MISSING, using empty')
        rows.append([case_id, '{}'])
        continue
    data_str = json.dumps(sol, separators=(',', ':'))
    n_blocks  = len(sol['outputs']['1 Block Design'])
    n_seqs    = len(sol['outputs']['2 Blocking Sequence'])
    print(f'[ID {case_id}] {tag}  blocks={n_blocks}  seqs={n_seqs}  len={len(data_str):,}')
    rows.append([case_id, data_str])

with open('submission.csv', 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerows(rows)

print(f'\nWrote submission.csv  ({sum(len(r[1]) for r in rows[1:])/1e6:.1f} MB)')

In [ ]:
# Download
from google.colab import files
files.download('submission.csv')

## 8. Training Curves (for paper)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, (layer, mult) in enumerate([('l1', 1.0), ('l2', 1.0), ('l3', 1.0)]):
    tag = f'{layer}_{mult}'
    log = train_logs.get(tag, [])
    if not log:
        continue
    eps       = [d['ep']       for d in log]
    bests     = [d['best']     for d in log]
    sel_loss  = [d['sel_loss'] for d in log]

    axes[0][col].plot(eps, bests)
    axes[0][col].set_title(f'{layer.upper()} ×{mult} — Best Stress')
    axes[0][col].set_xlabel('Episode'); axes[0][col].set_ylabel('Stress Score')
    axes[0][col].grid(True, alpha=0.3)

    axes[1][col].plot(eps, sel_loss, color='orange')
    axes[1][col].set_title(f'{layer.upper()} ×{mult} — Selection Loss')
    axes[1][col].set_xlabel('Episode'); axes[1][col].set_ylabel('Loss')
    axes[1][col].grid(True, alpha=0.3)

plt.suptitle('LNS-RL Training Curves')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Saved training_curves.png')